In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime, timedelta
import re
import time
import warnings
warnings.filterwarnings("ignore")

# ── 1. CONFIG ─────────────────────────────────────────────────────────────────

NEWS_API_KEY = "9f54cab10c1546cb8c41b4411389f1b9"   # free tier at newsapi.org
QUERIES      = ["stock market", "S&P 500", "federal reserve", "inflation", "recession"]
MARKET_TICKER = "^GSPC"            # S&P 500 index
DAYS_BACK    = 30                   # free NewsAPI tier: max 30 days
                                    # upgrade or use GDELT for longer windows (see note below)

# NOTE: For historical data beyond 30 days, use the GDELT Project (free, no key needed):
# https://api.gdeltproject.org/api/v2/doc/doc?query=...&mode=artlist&format=json
# Drop-in replacement function included at the bottom of this file.

# ── 2. FETCH HEADLINES ────────────────────────────────────────────────────────

def fetch_headlines(query, api_key, days_back=30, page_size=100):
    """Pull news headlines from NewsAPI for a given query."""
    from_date = (datetime.utcnow() - timedelta(days=days_back)).strftime("%Y-%m-%d")
    url = "https://newsapi.org/v2/everything"

    params = {
        "q":          query,
        "from":       from_date,
        "language":   "en",
        "sortBy":     "publishedAt",
        "pageSize":   page_size,
        "apiKey":     api_key,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    articles = []
    for a in data.get("articles", []):
        articles.append({
            "query":       query,
            "title":       a.get("title", ""),
            "description": a.get("description", ""),
            "source":      a.get("source", {}).get("name", ""),
            "published":   a.get("publishedAt", ""),
            "url":         a.get("url", ""),
        })

    return articles


all_articles = []
for q in QUERIES:
    print(f"Fetching: '{q}'...")
    try:
        articles = fetch_headlines(q, NEWS_API_KEY, days_back=DAYS_BACK)
        all_articles.extend(articles)
        time.sleep(0.5)   # be polite to the API
    except Exception as e:
        print(f"  Error fetching '{q}': {e}")

df = pd.DataFrame(all_articles)
df["published"] = pd.to_datetime(df["published"], utc=True, errors="coerce")
df = df.dropna(subset=["published"])
df["date"] = df["published"].dt.date

# deduplicate — same headline may appear under multiple queries
df = df.drop_duplicates(subset=["title"])
print(f"\nTotal unique headlines: {len(df)}")

# ── 3. CLEAN ──────────────────────────────────────────────────────────────────

FINANCE_TERMS = {
    r"\bbull(ish)?\b":  "optimistic market rising",
    r"\bbear(ish)?\b":  "pessimistic market falling",
    r"\brally\b":       "strong price increase",
    r"\bplunge[ds]?\b": "sharp price drop",
    r"\bsurge[ds]?\b":  "rapid price increase",
    r"\bslump[s]?\b":   "prolonged price decline",
    r"\bcrash(es|ed)?\b": "severe sudden market drop",
    r"\btumble[ds]?\b": "notable price decline",
    r"\bsoar(s|ed)?\b": "dramatic price rise",
    r"\bfed\b":         "federal reserve",
    r"\bhike[ds]?\b":   "interest rate increase",
}

def clean_headline(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    for pattern, replacement in FINANCE_TERMS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


# combine title + description for richer signal
df["full_text"] = (df["title"].fillna("") + " " + df["description"].fillna(""))
df["clean_text"] = df["full_text"].apply(clean_headline)

# ── 4. SENTIMENT ──────────────────────────────────────────────────────────────

analyzer = SentimentIntensityAnalyzer()

def score(text):
    s = analyzer.polarity_scores(text)
    label = "positive" if s["compound"] >= 0.05 else \
            "negative" if s["compound"] <= -0.05 else "neutral"
    return pd.Series({"compound": s["compound"], "sentiment": label})

df[["compound", "sentiment"]] = df["clean_text"].apply(score)
print(df["sentiment"].value_counts())

# ── 5. FETCH MARKET DATA ──────────────────────────────────────────────────────

start = df["published"].min().strftime("%Y-%m-%d")
end   = datetime.utcnow().strftime("%Y-%m-%d")

print(f"\nFetching {MARKET_TICKER} from {start} to {end}...")
market = yf.download(MARKET_TICKER, start=start, end=end, progress=False)
market = market[["Close"]].reset_index()
market.columns = ["date", "close"]
market["date"] = pd.to_datetime(market["date"]).dt.date
market["market_return"] = market["close"].pct_change() * 100

# daily average sentiment
daily_sentiment = (df.groupby("date")["compound"]
                     .mean()
                     .reset_index()
                     .rename(columns={"compound": "avg_sentiment"}))
daily_sentiment["date"] = pd.to_datetime(daily_sentiment["date"]).dt.date

merged = pd.merge(daily_sentiment, market, on="date", how="inner")
print(f"Merged dataset: {len(merged)} trading days")

# ── 6. VISUALISE ──────────────────────────────────────────────────────────────

palette = {"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"}
sns.set_theme(style="whitegrid", font_scale=1.1)

# — 6a. Sentiment distribution ————————————————————————————————————————————————

fig, ax = plt.subplots(figsize=(8, 5))
counts = df["sentiment"].value_counts().reindex(["positive", "neutral", "negative"])
bars = ax.bar(counts.index, counts.values,
              color=[palette[s] for s in counts.index],
              width=0.5, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(val), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_title("Sentiment Distribution — Financial News Headlines", fontsize=14, pad=12)
ax.set_ylabel("Number of headlines")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig1_sentiment_distribution.png", dpi=150)
plt.close()

# — 6b. Sentiment over time ———————————————————————————————————————————————————

merged_plot = merged.copy()
merged_plot["date"] = pd.to_datetime(merged_plot["date"])

fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.fill_between(merged_plot["date"], merged_plot["avg_sentiment"], 0,
                 where=merged_plot["avg_sentiment"] >= 0,
                 alpha=0.3, color="#2ecc71", label="Positive sentiment")
ax1.fill_between(merged_plot["date"], merged_plot["avg_sentiment"], 0,
                 where=merged_plot["avg_sentiment"] < 0,
                 alpha=0.3, color="#e74c3c", label="Negative sentiment")
ax1.plot(merged_plot["date"], merged_plot["avg_sentiment"],
         color="#2c3e50", linewidth=1.2, zorder=3)
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax1.set_ylabel("Avg sentiment score", color="#2c3e50")
ax1.set_xlabel("")

ax2 = ax1.twinx()
ax2.plot(merged_plot["date"], merged_plot["close"],
         color="#3498db", linewidth=1.5, alpha=0.7, linestyle="--", label="S&P 500")
ax2.set_ylabel("S&P 500 close", color="#3498db")

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45, ha="right")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc="upper left")

ax1.set_title("News Sentiment vs S&P 500 Price", fontsize=14, pad=12)
ax1.spines[["top"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig2_sentiment_vs_market.png", dpi=150)
plt.close()

# — 6c. Lag correlation: does sentiment today predict returns tomorrow? ————————

lag_results = []
for lag in range(0, 6):
    merged[f"lag_{lag}"] = merged["avg_sentiment"].shift(lag)
    corr = merged["market_return"].corr(merged[f"lag_{lag}"])
    lag_results.append({"lag_days": lag, "correlation": corr})

lag_df = pd.DataFrame(lag_results)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in lag_df["correlation"]]
ax.bar(lag_df["lag_days"], lag_df["correlation"], color=colors,
       edgecolor="white", linewidth=0.8, width=0.6)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Lag Correlation: News Sentiment → Next-Day Market Return", fontsize=14, pad=12)
ax.set_xlabel("Sentiment lag (days)")
ax.set_ylabel("Pearson correlation with market return")
ax.set_xticks(lag_df["lag_days"])
ax.set_xticklabels([f"Lag {d}" for d in lag_df["lag_days"]])
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig3_lag_correlation.png", dpi=150)
plt.close()

# — 6d. Sentiment by news source —————————————————————————————————————————————

source_df = (df.groupby("source")["compound"]
               .agg(["mean", "count"])
               .query("count >= 5")
               .sort_values("mean", ascending=False)
               .head(12)
               .reset_index())

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in source_df["mean"]]
ax.barh(source_df["source"], source_df["mean"], color=colors,
        edgecolor="white", linewidth=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Average Headline Sentiment by News Source", fontsize=14, pad=12)
ax.set_xlabel("Mean compound sentiment score")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("fig4_sentiment_by_source.png", dpi=150)
plt.close()

print("\nAll figures saved.")

# ── 7. SAVE ───────────────────────────────────────────────────────────────────

df.to_csv("news_sentiment_results.csv", index=False)
merged.to_csv("news_sentiment_market_merged.csv", index=False)
print("Results saved.")

# ── 8. GDELT ALTERNATIVE (no API key, longer history) ────────────────────────
#
# def fetch_gdelt(query, max_records=250):
#     url = "https://api.gdeltproject.org/api/v2/doc/doc"
#     params = {
#         "query":      query,
#         "mode":       "artlist",
#         "maxrecords": max_records,
#         "format":     "json",
#         "timespan":   "1y",   # up to full year of history
#     }
#     r = requests.get(url, params=params)
#     articles = r.json().get("articles", [])
#     return [{
#         "title":     a.get("title", ""),
#         "source":    a.get("domain", ""),
#         "published": a.get("seendate", ""),
#         "url":       a.get("url", ""),
#     } for a in articles]

: 